In [1]:
import sqlite3
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from category_encoders import OneHotEncoder
from IPython.display import VimeoVideo
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.utils.validation import check_is_fitted

warnings.simplefilter(action="ignore", category=FutureWarning)

In [ ]:
%load_ext sql
%sql sqlite:////home/jovyan/nepal.sqlite

In [ ]:
%%sql
SELECT * 
FROM household_demographics
LIMIT 5

household_id	gender_household_head	age_household_head	caste_household	education_level_household_head	income_level_household	size_household	is_bank_account_present_in_household
101	Male	31.0	Rai	Illiterate	Rs. 10 thousand	3.0	0.0
201	Female	62.0	Rai	Illiterate	Rs. 10 thousand	6.0	0.0
301	Male	51.0	Gharti/Bhujel	Illiterate	Rs. 10 thousand	13.0	0.0
401	Male	48.0	Gharti/Bhujel	Illiterate	Rs. 10 thousand	5.0	0.0
501	Male	70.0	Gharti/Bhujel	Illiterate	

In [ ]:
%%sql
SELECT count(*) 
FROM household_demographics
LIMIT 5

%%sql
SELECT count(*) 
FROM household_demographics
LIMIT 5

In [ ]:
%%sql
SELECT * 
FROM id_map
LIMIT 5


Running query in 'sqlite:////home/jovyan/nepal.sqlite'
household_id	building_id	vdcmun_id	district_id
5601	56	7	1
6301	63	7	1
9701	97	7	1
9901	99	7	1
11501	115	7	1


In [ ]:
%%sql
SELECT h.*,
       s.*,
       i.vdcmun_id,
       d.damage_grade
FROM household_demographics AS h
JOIN id_map AS i ON i.household_id =h.household_id
JOIN building_structure AS s ON i.building_id = s.building_id
JOIN building_damage AS d ON i.building_id = d.building_id
WHERE district_id = 4
LIMIT 5

household_id	gender_household_head	age_household_head	caste_household	education_level_household_head	income_level_household	size_household	is_bank_account_present_in_household	building_id	count_floors_pre_eq	count_floors_post_eq	age_building	plinth_area_sq_ft	height_ft_pre_eq	height_ft_post_eq	land_surface_condition	foundation_type	roof_type	ground_floor_type	other_floor_type	position	plan_configuration	condition_post_eq	superstructure	vdcmun_id	damage_grade
16400201	Female	46.0	Chhetree	Class 5	Rs. 10-20 thousand	4.0	1.0	164002	3	3	20	560	18	18	Flat	Mud mortar-Stone/Brick	Bamboo/Timber-Light roof	Mud	TImber/Bamboo-Mud	Not attached	Rectangular	Damaged-Repaired and used	Stone, mud mortar	38	Grade 2
16408101	Male	66.0	Chhetree	Illiterate	Rs. 10 thousand	5.0	0.0	164081	2	2	21	200	12	12	Flat	Mud mortar-Stone/Brick	Bamboo/Timber-Light roof	Mud	TImber/Bamboo-Mud	Not attached	Rectangular	Damaged-Used in risk	Stone, mud mortar	38	Grade 2
16408901	Male	54.0	Magar	Class 4	Rs. 10 thousand	5.0	1.0	164089	3	3	18	315	20	20	Flat	Mud mortar-Stone/Brick	Bamboo/Timber-Light roof	Mud	TImber/Bamboo-Mud	Not attached	Rectangular	Damaged-Used in risk	Stone, mud mortar	38	Grade 2
16409801	Male	36.0	Chhetree	Class 5	Rs. 10 thousand	6.0	1.0	164098	2	2	45	290	13	13	Flat	Mud mortar-Stone/Brick	Bamboo/Timber-Light roof	Mud	TImber/Bamboo-Mud	Not attached	Rectangular	Damaged-Used in risk	Stone, mud mortar	38	Grade 3
16410301	Female	39.0	Chhetree	Class 4	Rs. 10 thousand	

In [ ]:
def wrangle(db_path):
    # Connect to database
    conn = sqlite3.connect(db_path)

    # Construct query
    query = """SELECT h.*,
       s.*,
       i.vdcmun_id,
       d.damage_grade
FROM household_demographics AS h
JOIN id_map AS i ON i.household_id =h.household_id
JOIN building_structure AS s ON i.building_id = s.building_id
JOIN building_damage AS d ON i.building_id = d.building_id
WHERE district_id = 4
"""

    # Read query results into DataFrame
    df = pd.read_sql(query,conn, index_col="household_id")

    # Identify leaky columns
    drop_cols = [col for col in df.columns if "post_eq" in col]

    # Add high-cardinality / redundant column
    drop_cols.append("building_id")

    # Create binary target column
    df["damage_grade"] = df["damage_grade"].str[-1].astype(int)
    df["severe_damage"] = (df["damage_grade"] > 3).astype(int)

    # Drop old target
    drop_cols.append("damage_grade")

    # Drop multicollinearity column
    drop_cols.append("count_floors_pre_eq")
    
    # Drop columns
    df.drop(columns=drop_cols, inplace=True)

    return df

In [ ]:
df = wrangle("/home/jovyan/nepal.sqlite")
df.head()

In [17]:
#LOCAL WRANGLE
def wrangle():
    df = pd.read_csv("data/demographics.csv")

    # Set 'household_id' as index
    df.set_index("household_id", inplace=True)

    # Identify and remove leaky features
    drop_cols = [col for col in df.columns if "post_eq" in col]
    df.drop(columns=drop_cols, inplace=True)

    # Binary classification of target
    df['damage_grade'] = df["damage_grade"].str[-1].astype(int)
    df['severe_damage'] = (df['damage_grade'] > 3).astype(int)
    df.drop(columns=['damage_grade'], inplace=True)

    #Multicollinearity Columns drop
    df.drop(columns=['count_floors_pre_eq'], inplace=True)

    #High Cardinality
    df.drop(columns=['building_id'],inplace=True)


    return df

In [18]:
df = wrangle()
df.head()
df.columns

Index(['gender_household_head', 'age_household_head', 'caste_household',
       'education_level_household_head', 'income_level_household',
       'size_household', 'is_bank_account_present_in_household',
       'age_building', 'plinth_area_sq_ft', 'height_ft_pre_eq',
       'land_surface_condition', 'foundation_type', 'roof_type',
       'ground_floor_type', 'other_floor_type', 'position',
       'plan_configuration', 'superstructure', 'vdcmun_id', 'severe_damage'],
      dtype='object')

In [19]:
# Check your work
assert df.shape == (75883, 20), f"`df` should have shape (75883, 20), not {df.shape}"